# Assignment 3 — Retrieval-Augmented Generation (RAG): Machine Learning Knowledge Assistant

An end-to-end RAG application that answers Machine Learning questions **using only the content of a book**.

| Item | Choice |
|---|---|
| Source document | *Introduction to Machine Learning with Python* — Andreas C. Müller & Sarah Guido (`intro-to-ml-book.pdf`) |
| Framework | LangChain |
| Embedding model | Gemini `gemini-embedding-2` |
| Vector database | FAISS (local, in-memory, saved to disk) |
| LLM | Gemini `gemini-3.6-flash` |

**Notebook outline**

- **Part 1** — Data understanding & preprocessing (load, extract, explore, clean, chunk)
- **Part 2** — Embeddings & vector database
- **Part 3** — Retrieval pipeline (top-k experiments with k = 3, 5, 7)
- **Part 4** — Answer generation with a grounded prompt
- **Part 5** — End-to-end RAG application with real-time Q&A

## 0. Setup

In [ ]:
!pip install -q langchain langchain-google-genai langchain-core langchain-community langchain-text-splitters pypdf faiss-cpu

In [ ]:
import os        # environment variables and file paths
import re        # text cleaning
import bisect    # maps a chunk position back to its page number
import time      # pauses between embedding batches
import getpass   # lets us pass the API key without displaying it
import statistics

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API Key: ")

In [ ]:
LLM_MODEL = "gemini-3.6-flash"
EMBEDDING_MODEL = "gemini-embedding-2"

llm = ChatGoogleGenerativeAI(model=LLM_MODEL)
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

---
## Part 1 — Data Understanding & Preprocessing

### 1.1 Load the PDF document

In [ ]:
PDF_NAME = "intro-to-ml-book.pdf"

# Works in Google Colab (/content) and when run locally next to the PDF
candidates = [PDF_NAME, os.path.join("/content", PDF_NAME)]
PDF_PATH = next((p for p in candidates if os.path.exists(p)), None)

if PDF_PATH is None:
    try:
        from google.colab import files   # only available in Colab
        print(f"'{PDF_NAME}' not found - please upload it:")
        uploaded = files.upload()
        PDF_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Place '{PDF_NAME}' in the same folder as this notebook.")

print("Using PDF:", PDF_PATH)

In [ ]:
loader = PyPDFLoader(PDF_PATH)   # one Document per PDF page, with page metadata
documents = loader.load()

print("Total no of pages:", len(documents))
print("Metadata of first page:", documents[0].metadata)

### 1.2 Extract text from all pages

In [ ]:
book_text = ""
for page in documents:
    book_text += page.page_content + "\n"

print(f"Total characters : {len(book_text):,}")
print(f"Total words      : {len(book_text.split()):,}")
print(f"Approx. tokens   : {len(book_text) // 4:,}   (rule of thumb: 1 token ~ 4 characters)")

In [ ]:
# A sample from the first page of Chapter 1
chapter1_page = next(i for i, d in enumerate(documents) if d.page_content.strip().startswith("CHAPTER 1"))
print(f"PDF page {chapter1_page + 1}:\n")
print(documents[chapter1_page].page_content[:1200])

### 1.3 Explore the document structure

In [ ]:
page_lengths = [len(d.page_content) for d in documents]

print("Number of pages           :", len(documents))
print("Average characters / page :", round(statistics.mean(page_lengths)))
print("Median characters / page  :", round(statistics.median(page_lengths)))
print("Shortest / longest page   :", min(page_lengths), "/", max(page_lengths))

# Chapter start pages -> shows how the book is organised
print("\nBook structure (PDF page where each section starts):")
for i, d in enumerate(documents):
    text = d.page_content.strip()
    if re.match(r"(CHAPTER \d+|Preface|Index|Table of Contents)\b", text):
        title = " ".join(text.split("\n")[:3])[:70]
        print(f"  page {i + 1:>3} : {title}")

In [ ]:
# ---- Text quality checks ----
empty_pages   = [i + 1 for i, n in enumerate(page_lengths) if n < 50]
hyphen_breaks = len(re.findall(r"[‐­]\n(?=[a-z])", book_text))    # words split across lines (soft hyphen)
dot_leaders   = [i + 1 for i, d in enumerate(documents) if re.search(r"(\. ){5,}", d.page_content)]
footer_pages  = sum(1 for d in documents
                    if re.search(r"(^\d+\s*\|\s.+|.+\s\|\s*\d+)\s*$", d.page_content.strip().split("\n")[-1]))
code_pages    = sum(1 for d in documents if re.search(r"\b(In|Out)\[\d+\]:", d.page_content))

print("Blank / near-empty pages             :", len(empty_pages), empty_pages)
print("Words hyphenated across line breaks  :", hyphen_breaks)
print("Pages with table-of-contents dots    :", dot_leaders)
print("Pages ending with a running footer   :", footer_pages)
print("Pages containing code cells (In[n]:) :", code_pages)

In [ ]:
# Examples of the issues found above
m = re.search(r".{60}[‐­]\n[a-z].{40}", book_text)
print("Hyphenated line break :", repr(m.group(0)) if m else "none")

print("Running footer        :", repr(documents[chapter1_page + 1].page_content.strip().split("\n")[-1]))

if dot_leaders:
    print("Table of contents     :", repr(documents[dot_leaders[0] - 1].page_content[18:140]))

**Observations on structure and text quality**

- The book is a text-based PDF (not scanned), so `pypdf` extracts clean text and no OCR is needed.
- It is organised as front matter → Table of Contents → Preface → 8 chapters → Index → colophon.
- Issues that would hurt retrieval if left untreated:
  1. **Hyphenated line breaks** – words such as `pow‐\nerful` are split in two, so they no longer match the real word.
  2. **Running footers / page numbers** – every page ends with a line like `2 | Chapter 1: Introduction`, which is noise repeated across the book.
  3. **Table of Contents and Index pages** – long lists of headings/keywords with page numbers. They match almost any query by keyword but contain no explanation, so they pollute the search results.
  4. **Blank pages** and the cover / copyright / colophon pages carry no ML knowledge.
  5. **Figures, plots and equations** are images and are not extracted – only their captions and the surrounding text are available. Code cells (`In[n]:` / `Out[n]:`) are extracted as plain text.

The next step cleans items 1–4. Item 5 is a known limitation of a text-only pipeline.

### 1.4 Clean the text

In [ ]:
def clean_page_text(text: str) -> str:
    # Fix the text-quality issues found during exploration.
    text = re.sub(r"[‐­]\n(?=[a-z])", "", text)      # re-join words hyphenated across lines
    lines = [ln.rstrip() for ln in text.strip().split("\n")]
    # drop the running footer / page number on the last line
    if lines and re.match(r"^(\d+\s*\|\s.+|.+\s\|\s*\d+|\d+)$", lines[-1].strip()):
        lines = lines[:-1]
    text = "\n".join(lines)
    text = re.sub(r"(\. ){3,}\.?", " ", text)                  # table-of-contents dot leaders
    text = re.sub(r"[ \t]{2,}", " ", text)                     # repeated spaces
    text = re.sub(r"\n{3,}", "\n\n", text)                     # repeated blank lines
    return text.strip()


def find_page(docs, prefix, default):
    # index of the first page whose text starts with `prefix`
    return next((i for i, d in enumerate(docs) if d.page_content.strip().startswith(prefix)), default)


def load_and_clean_pdf(pdf_path: str):
    # Load a PDF, keep only the knowledge-bearing pages and clean their text.
    docs = PyPDFLoader(pdf_path).load()

    toc_page    = find_page(docs, "Table of Contents", 0)
    first_page  = next((i for i, d in enumerate(docs)                     # Preface (after the ToC)
                        if i > toc_page and d.page_content.strip().startswith("Preface")),
                       find_page(docs, "CHAPTER 1", 0))
    index_page  = next((i for i, d in enumerate(docs)                     # Index at the back
                        if i > first_page and re.match(r"Index\s*\n", d.page_content.strip())),
                       len(docs))

    cleaned = []
    for i, d in enumerate(docs):
        if not (first_page <= i < index_page):
            continue                                   # skip cover, ToC, index, colophon
        text = clean_page_text(d.page_content)
        if len(text) < 50:
            continue                                   # skip blank pages
        cleaned.append(Document(page_content=text,
                                metadata={"source": os.path.basename(pdf_path), "page": i + 1}))
    return cleaned


clean_documents = load_and_clean_pdf(PDF_PATH)

print("Pages before cleaning :", len(documents))
print("Pages after cleaning  :", len(clean_documents))
print("PDF pages kept        :", clean_documents[0].metadata["page"], "to", clean_documents[-1].metadata["page"])
print("Characters after clean:", f"{sum(len(d.page_content) for d in clean_documents):,}")

In [ ]:
# Before vs after for one page
sample = chapter1_page + 1
print("BEFORE (end of page):\n", documents[sample].page_content[-330:])
after = next(d for d in clean_documents if d.metadata["page"] == sample + 1)
print("\nAFTER (end of page):\n", after.page_content[-300:])

### 1.5 Split the text into chunks

The assignment suggests **500–1000 tokens per chunk with 50–150 tokens overlap**. `RecursiveCharacterTextSplitter` measures length in characters, so the sizes are converted using the rule of thumb *1 token ≈ 4 characters*.

- The cleaned pages are first joined into one continuous text. A page break in a PDF is arbitrary and often falls in the middle of an explanation, so splitting page by page would cut ideas in half and produce many small chunks.
- The splitter tries to cut at paragraph breaks first, then line breaks, then sentences, so chunks stay as coherent as possible.
- Every chunk records the PDF page it starts and ends on, which is used later to cite sources.

A few configurations are compared first.

In [ ]:
def make_chunks(docs, chunk_size, chunk_overlap):
    # Join the cleaned pages into one continuous text, so a chunk can run across a page break,
    # and remember where every page starts so each chunk can be traced back to its page(s).
    full_text, page_starts, page_numbers = "", [], []
    for d in docs:
        page_starts.append(len(full_text))
        page_numbers.append(d.metadata["page"])
        full_text += d.page_content + "\n"

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )
    pieces = splitter.create_documents([full_text])

    def page_at(offset):
        return page_numbers[max(bisect.bisect_right(page_starts, offset) - 1, 0)]

    for c in pieces:
        start = max(c.metadata.get("start_index", 0), 0)
        c.metadata = {
            "source": docs[0].metadata["source"],
            "page": page_at(start),                                   # page where the chunk starts
            "page_end": page_at(start + len(c.page_content) - 1),     # page where it ends
        }
    return pieces


def page_ref(meta) -> str:
    # "52" for a chunk on one page, "52-53" for a chunk that spans a page break
    first, last = meta["page"], meta.get("page_end", meta["page"])
    return str(first) if first == last else f"{first}-{last}"


print(f"{'chunk_size':>10} {'overlap':>8} {'#chunks':>8} {'avg chars':>10} {'avg ~tokens':>12} {'min':>6} {'max':>6}")
for size, overlap in [(1000, 200), (2000, 300), (3000, 400), (4000, 500)]:
    c = make_chunks(clean_documents, size, overlap)
    lens = [len(x.page_content) for x in c]
    print(f"{size:>10} {overlap:>8} {len(c):>8} {round(statistics.mean(lens)):>10} {round(statistics.mean(lens)) // 4:>12} {min(lens):>6} {max(lens):>6}")

**Chosen configuration: `chunk_size = 3000` characters (~750 tokens), `chunk_overlap = 400` characters (~100 tokens)**

- This sits in the middle of the suggested 500–1000 token range. A book page averages about 1,700 characters, so each chunk carries roughly one and a half to two pages: enough to keep an explanation together with its code example or its list of strengths and weaknesses.
- Small chunks (1000 characters, as in the class example) work for a short report, but here they split explanations mid-way and produce several times more vectors to embed.
- Very large chunks dilute the embedding (many topics in one vector) and send more irrelevant text to the LLM.
- The 400-character overlap repeats the last few sentences of a chunk at the start of the next, so an idea that falls on a boundary is still found.

In [ ]:
CHUNK_SIZE = 3000      # ~750 tokens
CHUNK_OVERLAP = 400    # ~100 tokens

chunks = make_chunks(clean_documents, CHUNK_SIZE, CHUNK_OVERLAP)

print("Total chunks:", len(chunks))
print("Average chunk length:", round(statistics.mean(len(c.page_content) for c in chunks)), "characters")
print("\nSample chunk metadata:", chunks[40].metadata)
print("\nSample chunk text:\n", chunks[40].page_content[:700])

---
## Part 2 — Embedding & Vector Database

### 2.1 Embedding model

**Model used: Google Gemini `gemini-embedding-2`** (through `GoogleGenerativeAIEmbeddings`).

- An embedding model turns a piece of text into a dense numeric vector so that texts with similar *meaning* get vectors that are close together, even when they use different words (for example "overfitting" and "model memorises the training data").
- The same model embeds both the book chunks (once, at indexing time) and the user query (at question time), so the two live in the same vector space and can be compared.
- It is a hosted model: no GPU or local model download is needed, and it accepts long inputs, which suits ~750-token chunks.

In [ ]:
sample_vector = embeddings.embed_query("What is overfitting?")

print("Embedding model     :", EMBEDDING_MODEL)
print("Embedding dimension :", len(sample_vector))
print("First 8 values      :", [round(v, 4) for v in sample_vector[:8]])

### 2.2 Generate embeddings for all chunks and store them in FAISS

**Vector storage approach: FAISS** (Facebook AI Similarity Search)

- FAISS keeps all chunk vectors in an in-memory index. LangChain's default is a *flat* index (`IndexFlatL2`): the query vector is compared against **every** stored vector using Euclidean (L2) distance. This is an exact search, and it is fast for a few hundred vectors.
- Alongside the index LangChain keeps a **docstore** that maps each vector back to its chunk text and metadata (source, page).
- The index is saved to disk with `save_local`, so the book is embedded only once. Later runs reload it instead of calling the embedding API again.
- No server or account is needed, which makes it a good fit for a single-book notebook application. For a much larger or shared corpus, a persistent database such as ChromaDB would be a better choice.

Chunks are embedded in small batches with a retry, because the free Gemini tier limits requests per minute.

In [ ]:
def with_retry(fn, attempts=6, first_wait=5):
    # Call fn(); on a temporary API error (429 / 503) wait and try again.
    wait = first_wait
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == attempts:
                raise
            print(f"   attempt {attempt} failed ({type(e).__name__}: {str(e)[:80]}) - retrying in {wait}s")
            time.sleep(wait)
            wait = min(wait * 2, 60)


def build_vector_store(chunks, embeddings, index_dir, batch_size=20, pause=1.0):
    # Embed all chunks and store them in FAISS. Reuses a saved index if present.
    if os.path.isdir(index_dir):
        print("Loading saved FAISS index from", index_dir)
        # the index was created by this notebook, so loading the pickle is safe
        return FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)

    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        vectors.extend(with_retry(lambda: embeddings.embed_documents(batch)))
        print(f"Embedded {min(start + batch_size, len(texts))}/{len(texts)} chunks")
        time.sleep(pause)

    store = FAISS.from_embeddings(
        text_embeddings=list(zip(texts, vectors)),
        embedding=embeddings,
        metadatas=metadatas,
    )
    store.save_local(index_dir)
    print("Saved FAISS index to", index_dir)
    return store


INDEX_DIR = f"faiss_index_ml_book_{CHUNK_SIZE}_{CHUNK_OVERLAP}"
vector_store = build_vector_store(chunks, embeddings, INDEX_DIR)

In [ ]:
print("Vectors stored in FAISS :", vector_store.index.ntotal)
print("Vector dimension        :", vector_store.index.d)
print("FAISS index type        :", type(vector_store.index).__name__)
print("Chunks in docstore      :", len(vector_store.index_to_docstore_id))

### 2.3 Similarity search — retrieved chunks for a sample query

In [ ]:
query = "What is the difference between supervised and unsupervised learning?"

results = vector_store.similarity_search(query=query, k=3)

for i, doc in enumerate(results):
    print(f"{i + 1}. [page {page_ref(doc.metadata)}]")
    print(doc.page_content[:600])
    print("-" * 100)

In [ ]:
# Same search with scores. The score is the L2 distance: LOWER = more similar.
scored_result = vector_store.similarity_search_with_score(query=query, k=3)

for i, (doc, score) in enumerate(scored_result):
    print(f"Result {i + 1}:")
    print(f"Score  : {score:.4f}")
    print(f"Page   : {page_ref(doc.metadata)}")
    print(f"Content: {doc.page_content[:300]}")
    print("-" * 50)

---
## Part 3 — Retrieval Pipeline

The retrieval function performs the steps explicitly:

1. **Accept** the user query
2. **Convert** the query into an embedding (same model as the chunks)
3. **Retrieve** the top-k closest chunks from FAISS

In [ ]:
def retrieve(query: str, k: int = 5, store=None):
    # Return the top-k (chunk, score) pairs for a query. Lower score = closer match.
    store = store or vector_store
    query_vector = with_retry(lambda: embeddings.embed_query(query))          # step 2: query -> embedding
    return store.similarity_search_with_score_by_vector(query_vector, k=k)    # step 3: top-k chunks


def show_results(results, preview=250):
    for i, (doc, score) in enumerate(results, start=1):
        text = " ".join(doc.page_content.split())
        print(f"  {i}. page {page_ref(doc.metadata):>7} | score {score:.4f} | {text[:preview]}...")


user_query = "How does a random forest reduce overfitting compared to a single decision tree?"
print("Query:", user_query, "\n")
show_results(retrieve(user_query, k=3))

### 3.1 Experiment with different values of k (3, 5, 7)

In [ ]:
test_queries = [
    "How does a random forest reduce overfitting compared to a single decision tree?",
    "What is the kernel trick in support vector machines?",
    "How does k-fold cross-validation work and why is it used?",
]

for q in test_queries:
    print("=" * 110)
    print("Query:", q)
    for k in (3, 5, 7):
        res = retrieve(q, k=k)
        pages = [page_ref(d.metadata) for d, _ in res]
        scores = [s for _, s in res]
        context_chars = sum(len(d.page_content) for d, _ in res)
        print(f"\n  k = {k} | pages {', '.join(pages)}")
        print(f"        | score best {min(scores):.4f} -> worst {max(scores):.4f} | context ~{context_chars // 4:,} tokens")

In [ ]:
# Look at what the extra chunks actually add when moving from k=3 to k=7
q = test_queries[0]
print("Query:", q, "\n")
show_results(retrieve(q, k=7), preview=160)

**How to read the k experiment**

- The results for k = 3 are always the first three results of k = 5 and k = 7 — a larger k only *adds* lower-ranked chunks. The question is whether the added chunks are still on topic.
- **k = 3** – smallest and most precise context. Good for a focused definition, but it can miss part of the answer when a topic is spread over several pages (for example an algorithm's explanation and its "strengths and weaknesses" section).
- **k = 5** – usually covers the neighbouring pages of the same section as well. Good balance between coverage and noise.
- **k = 7** – highest recall, but the last chunks have clearly worse (higher) distance scores and often come from other sections. The context sent to the LLM is more than twice the size of k = 3, which costs more and gives the model more off-topic text to be distracted by.

**Choice: k = 5 is used as the default** for answer generation. Compare the page lists and the score gap between the 3rd, 5th and 7th result printed above to confirm this on your run.

---
## Part 4 — Answer Generation (RAG)

### 4.1 Prompt design

The prompt is written so that the model answers **only from the retrieved chunks**:

| Prompt element | Why it is there |
|---|---|
| Role: "ML teaching assistant answering from the book" | Sets the tone and the scope of the task |
| "Use ONLY the context… do not use prior knowledge" | Stops the LLM answering from its own training data |
| Fixed refusal sentence when the context does not contain the answer | Gives the model a safe alternative to guessing, and makes refusals easy to detect |
| Each chunk is labelled `[Source n \| page p]` and the model must cite `[Source n]` | Makes every claim traceable to a chunk and page — evidence of grounding |
| "Supporting Evidence" section with short quotes | Forces the model to point at the exact text it used |
| Context placed before the question, question last | The model reads the evidence first, then the task |

In [ ]:
NOT_FOUND = "I couldn't find this information in the book."

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Machine Learning teaching assistant. You answer questions using the book "
     "'Introduction to Machine Learning with Python'.\n\n"
     "Rules:\n"
     "1. Use ONLY the information in the retrieved context below. Do not use prior knowledge.\n"
     "2. If the context does not contain the answer, reply exactly: \"" + NOT_FOUND + "\" "
     "and nothing else. Do not try to make up an answer.\n"
     "3. After every claim, cite the source it came from in the form [Source n].\n"
     "4. Be clear and concise. Explain the concept the way the book does."),
    ("human",
     "Retrieved context:\n{context}\n\n"
     "Question: {question}\n\n"
     "Provide:\n"
     "1. Answer (with [Source n] citations)\n"
     "2. Supporting Evidence: one or two short quotes copied from the context, each with its source number and page"),
])

# .with_retry() repeats the call if Gemini returns a temporary error such as "503 model overloaded"
rag_chain = rag_prompt | llm.with_retry(stop_after_attempt=5) | StrOutputParser()

### 4.2 Pass the retrieved chunks to the LLM as context

In [ ]:
def format_context(results) -> str:
    # Label every retrieved chunk so the LLM can cite it.
    blocks = []
    for i, (doc, _score) in enumerate(results, start=1):
        blocks.append(f"[Source {i} | page {page_ref(doc.metadata)}]\n{doc.page_content}")
    return "\n\n".join(blocks)


def ask(question: str, k: int = 5, store=None) -> dict:
    # Full RAG step: retrieve -> build context -> generate a grounded answer.
    results = retrieve(question, k=k, store=store)
    answer = rag_chain.invoke({"context": format_context(results), "question": question})
    sources = [{"source": i, "page": page_ref(d.metadata), "score": float(s), "text": d.page_content}
               for i, (d, s) in enumerate(results, start=1)]
    return {"question": question, "answer": answer, "sources": sources}


def show_answer(result: dict, show_sources: bool = True):
    print("QUESTION:", result["question"])
    print("\n" + result["answer"])
    if show_sources:
        print("\nRETRIEVED SOURCES:")
        for s in result["sources"]:
            preview = " ".join(s["text"].split())[:110]
            print(f"  [Source {s['source']}] page {s['page']:>7} | score {s['score']:.4f} | {preview}...")
    print("=" * 110)

### 4.3 Sample queries and generated answers

In [ ]:
question = "What is overfitting and how can it be detected?"

response = ask(question, k=5)
show_answer(response)

In [ ]:
sample_questions = [
    "What is the difference between supervised and unsupervised learning?",
    "How does the k-nearest neighbors algorithm make a prediction?",
    "What are the strengths and weaknesses of random forests?",
    "Why is feature scaling important for support vector machines?",
    "What is the difference between L1 and L2 regularization in linear models?",
    "What does a Pipeline do in scikit-learn and why is it useful with cross-validation?",
]

sample_results = []
for q in sample_questions:
    r = ask(q, k=5)
    sample_results.append(r)
    show_answer(r)

### 4.4 Evidence of grounding

Three checks show that the answers come from the retrieved context and not from the LLM's own knowledge.

**Check 1 — Answer next to its evidence.** The citations in the answer are matched to the retrieved chunks, and the cited chunk text is printed so the answer can be verified against the book.

In [ ]:
def cited_sources(result: dict):
    # Source numbers the answer cites, e.g. [Source 2] or [Source 1, Source 3]
    cited = set()
    for group in re.findall(r"\[([^\]]*Source[^\]]*)\]", result["answer"]):
        cited.update(int(n) for n in re.findall(r"\d+", group.split("|")[0]))
    return sorted(cited)


r = sample_results[2]     # the random forest question
cited = cited_sources(r)
valid = [s["source"] for s in r["sources"]]

print("QUESTION:", r["question"])
print("Sources cited in the answer :", cited)
print("Sources that were retrieved :", valid)
print("All citations are valid     :", bool(cited) and all(c in valid for c in cited))

for s in r["sources"]:
    if s["source"] in cited[:2]:
        print(f"\n--- Text of [Source {s['source']}] (page {s['page']}) ---")
        print(s["text"][:900])

**Check 2 — Questions the book cannot answer.** The first question is unrelated to the book. The other two are real ML/AI topics that the LLM knows well but that this 2016 book does not cover. A grounded system must refuse all three instead of answering from memory.

In [ ]:
out_of_scope_questions = [
    "Who won the FIFA World Cup in 2022?",
    "How does the self-attention mechanism in the Transformer architecture work?",
    "What is Retrieval-Augmented Generation and which vector databases does the book recommend for it?",
]

for q in out_of_scope_questions:
    r = ask(q, k=5)
    refused = NOT_FOUND.lower()[:35] in r["answer"].lower()
    show_answer(r, show_sources=False)
    print(">>> Refused to answer (grounded behaviour):", refused, "\n")

**Check 3 — Citation check across all sample answers.** Every answer should cite at least one source, and only sources that were actually retrieved.

In [ ]:
print(f"{'cited':<18} {'valid':<6} question")
for r in sample_results:
    cited = cited_sources(r)
    valid = {s["source"] for s in r["sources"]}
    ok = bool(cited) and set(cited) <= valid
    print(f"{str(cited):<18} {str(ok):<6} {r['question']}")

### 4.5 Effect of k on the generated answer

In [ ]:
question = "What are the strengths and weaknesses of random forests?"

for k in (3, 5, 7):
    r = ask(question, k=k)
    print(f"########## k = {k} | pages used: {', '.join(s['page'] for s in r['sources'])} | answer length: {len(r['answer'].split())} words")
    print(r["answer"])
    print()

---
## Part 5 — End-to-End RAG Application

All the steps are combined into one class:

```
PDF  ->  load & clean  ->  chunk  ->  embed  ->  FAISS index          (done once, in from_pdf)
question  ->  embed query  ->  top-k chunks  ->  grounded prompt  ->  LLM  ->  answer + sources   (every call to ask)
```

In [ ]:
class MLKnowledgeAssistant:
    # End-to-end RAG question-answering system over a machine learning book.

    def __init__(self, vector_store, k: int = 5):
        self.vector_store = vector_store
        self.k = k

    @classmethod
    def from_pdf(cls, pdf_path, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, k=5):
        # Document processing + indexing: PDF -> clean pages -> chunks -> embeddings -> FAISS
        pages = load_and_clean_pdf(pdf_path)
        pieces = make_chunks(pages, chunk_size, chunk_overlap)
        index_dir = f"faiss_index_ml_book_{chunk_size}_{chunk_overlap}"
        store = build_vector_store(pieces, embeddings, index_dir)
        print(f"Assistant ready: {len(pages)} pages -> {len(pieces)} chunks -> {store.index.ntotal} vectors")
        return cls(store, k=k)

    def ask(self, question: str, k: int = None) -> dict:
        # Retrieval + generation for one question
        return ask(question, k=k or self.k, store=self.vector_store)

    def chat(self):
        # Real-time question answering loop
        print("ML Knowledge Assistant - ask a question about the book (type 'exit' to stop)\n")
        while True:
            question = input("You: ").strip()
            if question.lower() in ("exit", "quit", "q", ""):
                print("Goodbye!")
                break
            result = self.ask(question)
            pages = ", ".join(s["page"] for s in result["sources"])
            print("\nAssistant:\n" + result["answer"])
            print(f"\n(Retrieved from PDF pages: {pages})\n" + "-" * 110)


# Builds the whole pipeline from the PDF (reuses the FAISS index saved in Part 2)
assistant = MLKnowledgeAssistant.from_pdf(PDF_PATH)

### 5.1 Demonstration with multiple queries

In [ ]:
demo_questions = [
    "Why do we split data into a training set and a test set?",
    "How does PCA reduce the number of features?",
    "When should I use k-means and what are its limitations?",
    "What is grid search and how is it combined with cross-validation?",
    "How does the bag-of-words representation turn text into features?",
    "Why is accuracy a poor metric for imbalanced datasets and what can be used instead?",
    "What does the book say about quantum machine learning?",      # not covered -> should refuse
]

for q in demo_questions:
    show_answer(assistant.ask(q))

### 5.2 Real-time question answering

Run the cell below and type your own questions. Type `exit` to stop.

In [ ]:
assistant.chat()

---
## Summary

| Part | What was done |
|---|---|
| 1. Preprocessing | Loaded the 392-page PDF with `PyPDFLoader`, explored its structure and text quality, removed front matter / ToC / index / blank pages, fixed hyphenation and footers, and split it into ~750-token chunks with ~100-token overlap |
| 2. Embedding & vector DB | Embedded every chunk with Gemini `gemini-embedding-2` and stored the vectors in a FAISS flat L2 index saved to disk |
| 3. Retrieval | Query → embedding → top-k chunks, compared k = 3, 5, 7 and chose k = 5 |
| 4. Generation | Grounded prompt with source labels, mandatory citations and a fixed refusal sentence; verified grounding with citation checks and out-of-scope questions |
| 5. Application | `MLKnowledgeAssistant` class that goes from PDF to answers, demonstrated on multiple queries and an interactive chat loop |

**Limitations and possible improvements**

- Figures, plots and equations in the book are not extracted, so questions that depend on a figure can only be answered from its caption and surrounding text.
- Retrieval is purely semantic. Adding keyword search (hybrid retrieval) or a re-ranker would help with exact terms such as scikit-learn parameter names.
- The refusal behaviour depends on the LLM following the prompt. An automatic faithfulness evaluation (for example with RAGAS) would measure hallucination more rigorously.